# Risk Analytics Spark Connect + Nessie Query Notebook

This notebook connects to Spark Connect and runs queries against Iceberg tables in the Nessie catalog.

It includes:
- Reading data from `nessie.risk_analytics.risk_metrics`
- Reading Iceberg snapshots metadata
- Listing Nessie branches/references

In [ ]:
import os

from pyspark.sql import SparkSession

# Use SPARK_REMOTE if provided; otherwise default to host mapping
spark_remote = os.getenv("SPARK_REMOTE", "sc://localhost:15002")
print(f"Using Spark Connect endpoint: {spark_remote}")

spark = SparkSession.builder.remote(spark_remote).appName("risk-analytics-spark-connect-queries").getOrCreate()
print("Spark session created")

In [ ]:
# Show available catalogs and confirm current context
spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SELECT current_catalog() AS current_catalog").show(truncate=False)

In [ ]:
# Query the published Iceberg risk metrics table
risk_df = spark.sql("""
SELECT
  as_of_date,
  customer_id,
  netting_set_id,
  gross_exposure,
  netting_exposure,
  collateral_value_after_haircut,
  pfe,
  var,
  source_branch,
  calculation_timestamp
FROM nessie.risk_analytics.risk_metrics
ORDER BY calculation_timestamp DESC
LIMIT 50
""")

risk_df.show(truncate=False)

In [ ]:
# Iceberg snapshots metadata table for audit/history
snapshots_df = spark.sql("""
SELECT
  snapshot_id,
  parent_id,
  committed_at,
  operation,
  manifest_list,
  summary
FROM nessie.risk_analytics.risk_metrics.snapshots
ORDER BY committed_at DESC
""")

snapshots_df.show(truncate=False)

In [ ]:
# Try to list Nessie references (branches/tags) through Spark SQL
try:
    spark.sql("SHOW REFERENCES IN nessie").show(truncate=False)
except Exception as e:
    print("SHOW REFERENCES is not supported in this Spark/Nessie combination.")
    print(f"Details: {e}")

In [ ]:
# Fallback/explicit branch listing via Nessie REST API
import requests

nessie_uri = os.getenv("NESSIE_URI", "http://localhost:19120/api/v2").rstrip("/")
resp = requests.get(f"{nessie_uri}/trees", timeout=10)
resp.raise_for_status()
refs = resp.json().get("references", [])

print(f"Nessie URI: {nessie_uri}")
print(f"Total references: {len(refs)}")
for ref in refs:
    print(f"- {ref.get('type')} {ref.get('name')} @ {ref.get('hash')}")

In [ ]:
# Optional: stop Spark session when done
# spark.stop()